# Caracterización de Punto de Operación (.OP), Respuesta AC y Área de la OTA (Sky130)
---
Este cuaderno realiza la caracterización integral del Amplificador Operacional de Transconductancia (**OTA**) a partir de los netlists y simulaciones guardados en `/headless/.xschem/simulations/`:

1. **Lector Nativo en Python Estándar de Archivos RAW:**
   - Carga y procesamiento directo de `OTA_ac.raw` y `OTA_op.raw`.

2. **Análisis Frecuencial y Margen de Fase (Respuesta AC):**
   - Interpolación lineal exacta de **Ganancia DC ($A_{v0}$)**, **Frecuencia de Corte ($f_{-3dB}$)**, **Frecuencia de Ganancia Unitaria ($GBW$)** y **Margen de Fase ($PM$)**.
   - Diagrama de Bode interactivo con Plotly aplicando **escala logarítmica explícita en ambos subgráficos** (magnitud y fase).

3. **Tabla de Punto de Operación (.OP) y Regiones de Trabajo:**
   - Extracción de parámetros de polarización ($g_m, g_{ds}, I_D, V_{GS}, V_{DS}, V_{th}, V_{dsat}$) para todos los transistores.

4. **Desglose Dinámico de Área en Silicio:**
   - Análisis dinámico del netlist `/headless/.xschem/simulations/OTA.spice` para extraer las dimensiones exactas $W, L, nf, mult$ de todos los componentes.



In [1]:
# 1. Definir las rutas (reemplaza con tus directorios exactos)
netlist = "/headless/.xschem/simulations/OTA_tb.spice"
raw_path = "/headless/.xschem/simulations/"

In [2]:
import site
import sys
import subprocess



# 1. Registrar la ruta de paquetes del usuario en el sistema
user_site = '/headless/.local/lib/python3.12/site-packages'
site.addsitedir(user_site)

# 2. Intentar importar; si falla, instalarlo automáticamente
try:
    from PyLTSpice import RawRead
    print("¡PyLTSpice ya estaba instalado y listo para usar!")
except ImportError:
    print("PyLTSpice no fue encontrado. Instalando automáticamente...")
    
    # Ejecuta pip install dentro del mismo Kernel de Python
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--user", "PyLTSpice"])
    
    # Re-asegurar que la ruta esté cargada
    site.addsitedir(user_site)
    
    # Importación tras la instalación
    from PyLTSpice import RawRead

¡PyLTSpice ya estaba instalado y listo para usar!


In [3]:
import pandas as pd
import numpy as np
from PyLTSpice import RawRead
import os
import re
from matplotlib.ticker import EngFormatter  # La librería mágica

def generar_tabla_inteligente_op(ruta_raw):
    """
    Lee un archivo OP, agrupa los datos por transistor, filtra la basura,
    y calcula ro, gm/Id y fT automáticamente.
    """
    if not os.path.exists(ruta_raw):
        print(f"⚠️ El archivo no existe: {ruta_raw}")
        return None
        
    try:
        raw_op = RawRead(ruta_raw)
    except Exception as e:
        print(f"⚠️ Error al leer el archivo RAW: {e}")
        return None

    variables = raw_op.get_trace_names()
    datos_agrupados = {}

    patron = r"x([a-zA-Z0-9_]+)\.msky130.*?\[([a-zA-Z0-9_]+)\]"

    for var in variables:
        match = re.search(patron, var, re.IGNORECASE)
        if match:
            transistor = match.group(1).upper() 
            parametro = match.group(2).lower()  
            
            wave = raw_op.get_trace(var).get_wave()
            if len(wave) > 0:
                valor = float(np.real(wave[0]))
                
                if parametro in ['id', 'vgs', 'vth', 'vdsat']:
                    valor = abs(valor)
                    
                if transistor not in datos_agrupados:
                    datos_agrupados[transistor] = {}
                    
                datos_agrupados[transistor][parametro] = valor

    # ==============================================================================
    # CÁLCULOS INTELIGENTES
    # ==============================================================================
    for t, params in datos_agrupados.items():
        if 'gds' in params and params['gds'] != 0:
            params['ro'] = 1 / params['gds']
            
        if 'gm' in params and 'id' in params and params['id'] != 0:
            params['gm/id'] = params['gm'] / params['id']
            
        if 'gm' in params and 'cgg' in params and params['cgg'] != 0:
            # EL ÚNICO CAMBIO: Le quitamos el "/ 1e9" para que la librería ponga la 'G' sola
            params['ft'] = params['gm'] / (2 * np.pi * params['cgg'])

    df = pd.DataFrame.from_dict(datos_agrupados, orient='index')
    
    # Le quitamos el (GHz) al nombre de la columna
    columnas_ideales = ['id', 'vgs', 'vth', 'vdsat', 'gm', 'ro', 'gm/id', 'cgg', 'ft']
    columnas_finales = [c for c in columnas_ideales if c in df.columns]
    df = df[columnas_finales]
    
    return df


def run_simulation(netlist_path,raw_pad):
  
    # ==========================================================================
    # 4. SIMULAR CON NGSPICE
    # ==========================================================================
    entorno = os.environ.copy()
    # Prepara la lista de palabras para el comando de terminal que ejecutará Ngspice.
    comando = ['/foss/tools/ngspice/bin/ngspice', "-b", netlist_path]
    
    # Ejecuta el comando en la terminal, atrapa los textos generados y se ubica en la carpeta del diseño.
    resultado = subprocess.run(comando, capture_output=True, text=True, cwd=raw_path)
    
    # Verifica si Ngspice terminó con un error interno (código diferente de cero).
    if resultado.returncode != 0:
        # Imprime en la pantalla que hubo un error.
        print("⚠️ La simulación falló. Aquí está el error de Ngspice:\n")
        # Imprime el mensaje de error exacto que arrojó Ngspice para que puedas leerlo.
        print(resultado.stderr)
    else:
        # Si el código fue cero, imprime un mensaje de éxito.
        print("¡Simulación terminada con éxito!")

    # ==========================================================================
    # 5. BORRAR EL ARCHIVO TEMPORAL
    # ==========================================================================
    
    # Revisa si el archivo temporal existe físicamente en el disco duro.
    


    return resultado.stdout
    


    

In [4]:
import subprocess
from PyLTSpice import RawRead
import time
import os 
import subprocess
import re


mi_log = run_simulation(netlist,raw_path)

¡Simulación terminada con éxito!


In [4]:
#"/headless/Desktop/simulations/LDO_op.raw")
import subprocess
from PyLTSpice import RawRead
import time
import os 
import subprocess
import re

ruta_op = os.path.join("/headless/Desktop/simulations/LDO_op.raw")#raw_path, "OTA_op.raw")
tabla_resultados = generar_tabla_inteligente_op(ruta_op)

# Creamos la máscara visual (places=2 son los decimales)
formateador_si = EngFormatter(places=2)

# Se la aplicamos a la tabla de Pandas
tabla_bonita = tabla_resultados.style.format(lambda x: formateador_si(x) if pd.notna(x) else "-")

display(tabla_bonita)

,id,vgs,vth,vdsat,gm,ro,gm/id
MN,19.68 µ,525.70 m,354.99 m,196.77 m,179.32 µ,169.88 k,9.11
MNR,19.68 µ,757.52 m,554.88 m,207.56 m,174.74 µ,582.27 k,8.88
MNREF,18.21 µ,728.83 m,526.11 m,184.45 m,179.77 µ,1.20 M,9.87
MO,14.64 µ,723.88 m,472.93 m,203.79 m,132.81 µ,1.75 M,9.07
MP,19.62 µ,524.37 m,354.99 m,195.82 m,179.80 µ,204.42 k,9.16
MPR,19.62 µ,757.52 m,554.88 m,207.56 m,174.24 µ,559.59 k,8.88
MR,18.22 µ,340.04 m,121.82 m,211.90 m,146.19 µ,144.01 k,8.03
MRC1,39.30 µ,340.04 m,121.82 m,211.90 m,315.91 µ,217.24 k,8.04
MRC2,14.64 µ,340.04 m,92.29 m,224.25 m,93.83 µ,18.79 k,6.41
M1,100.40 µ,146.65 m,79.85 m,106.88 m,1.29 m,56.49 k,12.80


In [6]:
import re


# 2. Crear un diccionario vacío para guardar los datos
parametros_ldo = {}

# 3. Expresión regular que busca "cualquier_palabra = numero"
# Soporta decimales (72.8159) y notación científica (1.00669e+06)
patron = r"([a-zA-Z0-9_]+)\s*=\s*([+-]?\d+\.?\d*[eE]?[+-]?\d*)"

# 4. Encontrar todas las coincidencias en el texto de mi_log
coincidencias = re.findall(patron, mi_log)
#print(coincidencias)
# 5. Llenar el diccionario convirtiendo el texto a números reales (float)
for nombre, valor in coincidencias:
    # nombre.strip() quita espacios extra por seguridad
    parametros_ldo[nombre.strip().lower()] = float(valor)


print(parametros_ldo)

# ¡Listo! Ahora puedes usar tus medidas del LDO como variables de Python
print("--- Datos extraídos del LOG ---")
print(f"Margen de fase: {parametros_ldo.get('margen_fase')} grados")
print(f"Ganancia UGB: {parametros_ldo.get('ganancia_banda_plana')} DB")




{'temp': 27.0, 'tnom': 27.0, 'ugb': 29357500.0, 'fase_ugb': -158.74, 'ganancia_banda_plana': 68.489, 'margen_fase': 21.2596}
--- Datos extraídos del LOG ---
Margen de fase: 21.2596 grados
Ganancia UGB: 68.489 DB
